# Parallel Processing

Some of the computational tasks performed in scqubits can benefit significantly from parallelization. The scqubits package leverages parallel-processing capabilities provided by the Python Standard Library `multiprocessing` module. For better pickling support, scqubits further supports use of `pathos` and `dill`.

One important consideration for parallelization of tasks like parameter sweeps is the fact that Numpy and Scipy tend to make use of multi-threading internally (through their BLAS backend). With several worker processes each spawning a full BLAS thread pool, the cores become oversubscribed and a sweep can run *slower* than with fewer threads per worker.

scqubits handles this for you: when `num_cpus > 1`, it caps each worker's BLAS threads automatically (via the `MULTIPROC_BLAS_THREADS` setting, default `"auto"` -- see *Limiting BLAS threads per worker* below), so in the common case there is nothing to configure. The sections below explain how to enable parallelization, when it actually helps, and the knobs available if you want to tune things by hand.

In [7]:
import numpy as np

import scqubits
from scqubits import HilbertSpace, InteractionTerm, ParameterSweep

## Quick start

Parallelization is opt-in, and scqubits can size it to the job for you. The shortest paths:

| If you want… | Do this |
|---|---|
| scqubits to choose the settings | pass `num_cpus="auto"` — it sizes the job, picks the workers, and stays serial when parallel wouldn't help |
| the best choices for *your* machine | run `scqubits.calibrate_parallelization()` once — it times your machine and saves the numbers `"auto"` then uses (re-running overwrites them, so you can recalibrate anytime) |
| every sweep auto-tuned, without a kwarg | set `scqubits.settings.AUTO_PARALLEL = True` |
| a fixed number of workers | pass `num_cpus=N` |
| to run a plain `.py` script with parallelism | guard the entry point with `if __name__ == "__main__":` |

Each row is explained in the sections below; the difference between `"auto"` and
`AUTO_PARALLEL` is covered under *Letting scqubits choose the settings*.

## Enabling parallel processing

Parallel processing is enabled for appropriate scqubits methods and classes by passing the number of cores to be used through the keyword argument `num_cpus`. The following classes and class methods support parallelization:

### Classes and class methods supporting parallelization

| Class or class method                          |
|------------------------------------------------|
| ``ParameterSweep``                             |
| ``HilbertSpace.get_spectrum_vs_paramvals``     |
| ``<qubit_class>.get_spectrum_vs_paramvals``    |
| ``<qubit_class>.plot_evals_vs_paramvals``      |
| ``<qubit_class>.get_matelements_vs_paramvals`` |
| ``<qubit_class>.plot_matelem_vs_paramvals``    |


To use parallelization, the keyword argument `num_cpus` must be passed, specifying the number of cores to be used as an integer, e.g.

In [ ]:
transmon.get_spectrum_vs_paramvals(..., num_cpus=4)

In [ ]:
sweep = ParameterSweep(
    param_name=param_name,
    ...,
    ...,
    num_cpus=4
)

Once `num_cpus` exceeds the value 1 when passed, scqubits starts a parallel processing pool of the desired number of processes.

## When does `num_cpus > 1` actually help?

Parallelization is **not free**, and for many sweeps it gives no speedup -- or even a
slowdown. Each grid point is shipped to a worker process (pickling + dispatch), and that
fixed overhead is only worth paying when there is enough work to amortize it:

> `num_cpus > 1` helps only when **(number of grid points) × (cost per point)&nbsp;≫&nbsp;the per-task overhead**.

What to expect, therefore:

- **Small grids, or cheap-per-point systems** (small Hilbert spaces, few eigenstates):
  `num_cpus > 1` gives little or no benefit, and is frequently *slower* than serial. This
  is the common case and is entirely normal -- keep the default `num_cpus = 1`.
- **Large grids of expensive points** (large composite Hilbert spaces, many grid points):
  parallel workers pay off.

If a `num_cpus` comparison looks 'inconclusive' or backwards (e.g. `num_cpus=2` slower
than `num_cpus=1`), the sweep is most likely below this break-even. (Oversubscription of
the cores by BLAS threads, historically the other common cause, is avoided by default --
see `MULTIPROC_BLAS_THREADS` below -- unless you have explicitly set it to `None`.) For a
hands-on demonstration of both regimes, see the `demo_multiprocessing` notebook in the
[scqubits-examples](https://github.com/scqubits/scqubits-examples) repository. Rather than
judging the break-even by hand, you can let scqubits pick `num_cpus` for you -- see
*Letting scqubits choose the settings* just below.

For large composite systems the per-point **diagonalization method** is often a bigger
lever than parallelism: sparse diagonalization (the default for large spectra; see
`AUTO_SPARSE_DIAG`) can be far faster per point, and once each point is cheap, `num_cpus >
1` helps even less. Try sparse first; parallelize second.

## Letting scqubits choose the settings

**A 30-second mental model.** There are two *separate* pieces, and they work together:

- **The switch — *whether* to parallelize.** That is `num_cpus`: leave it at the default (serial), set a number yourself, or say `"auto"` to let scqubits decide.
- **The map — *how well* `"auto"` decides.** `"auto"` always works, using built-in rules of thumb. Running `calibrate_parallelization()` once replaces those rules of thumb with real measurements of *your* machine, so `"auto"` decides better.

The calibration is **just data** — it does nothing by itself; it only sharpens the choices `"auto"` makes. So "everything optimal" needs **both**: turn on `"auto"` *and* (recommended) calibrate once. The reverse is fine too — `"auto"` works without calibrating, just with generic instead of machine-specific numbers.

### Asking for a recommendation

`scqubits.recommend_parallelization` is a *pure* function — it starts no worker processes, so it is safe to call anywhere — that reads the Hilbert-space dimension, the number of grid points, the eigenvalue count, and whether sparse diagonalization applies, and returns a recommendation:

```python
cfg = scqubits.recommend_parallelization(hilbertspace=hs, num_points=384, evals_count=20)
print(cfg.num_cpus, cfg.blas_threads, cfg.reason)
sweep = scqubits.ParameterSweep(..., num_cpus=cfg.num_cpus)
```

Because a `ParameterSweep` runs as soon as it is constructed, call `recommend_parallelization` *before* building the sweep. More conveniently, pass the sentinel `num_cpus="auto"`, which makes the sweep tune itself **before** it runs:

```python
sweep = scqubits.ParameterSweep(..., num_cpus="auto")
```

### `num_cpus="auto"` vs `settings.AUTO_PARALLEL = True` — same engine, different reach

Both trigger the *exact same* auto-tuner; the only difference is *when* it kicks in:

- **`num_cpus="auto"` — per call.** You opt in for one specific sweep; nothing else is affected.
- **`settings.AUTO_PARALLEL = True` — global default.** Now *any* sweep where you do **not** pass `num_cpus` behaves as if you had written `"auto"`. Set it once and forget it.

An explicit number always wins:

```text
num_cpus=4        ->  exactly 4 workers      (you decide; auto-tuner not consulted)
num_cpus="auto"   ->  auto-tuner decides     (always; calibrated if you ran calibrate())
num_cpus omitted  ->  auto-tuner if AUTO_PARALLEL=True, otherwise serial (the default)
```

The recommendation applies its choice live (no kernel restart) and works the same in Jupyter and in a plain script; only a sweep that the heuristic decides to parallelize starts workers, which in a plain script needs the `__main__` guard described below.

### Calibrating to your machine (recommended, run once)

`calibrate_parallelization()` times your hardware — how long it takes to start worker processes, and how expensive a grid point is to diagonalize (dense and sparse) — and saves the result to a small JSON file (`~/.scqubits/parallel_calibration.json` by default; change the location with `settings.PARALLEL_CALIBRATION_PATH`). From then on, every `num_cpus="auto"` decision reads that file and tailors its choice to *your* machine instead of using generic defaults.

```python
scqubits.calibrate_parallelization()   # ~1 minute; measures this machine and writes the file
```

**Calibrate under realistic, steady conditions — and redo it freely.** The measurement is only as good as the machine state while it runs, and **re-running simply overwrites the previous file**, so recalibrating is cheap and safe. Redo it whenever:

- you accidentally calibrated while the machine was **busy** with other work (the measured costs come out inflated), or
- a **laptop was on battery / CPU-throttled** at calibration time — many laptops clock down sharply when unplugged, so the calibration over-estimates every cost and `"auto"` then plays it too safe (parallelizes less than it should), or
- you **changed hardware**.

For the most representative numbers, calibrate on an otherwise-idle machine, plugged into wall power. The calibration runs its measurements as `python -m` subprocesses, so the call itself needs no `__main__` guard.

> Tip: if `"auto"` ever seems oddly conservative, your calibration may have been taken under load or on battery — just run `calibrate_parallelization()` again on a quiet, plugged-in machine to refresh it.

## Global num_cpus default

The global default for `num_cpus` is stored in `scqubits.settings.NUM_CPUS`. Upon import of scqubits, that constant has the value `1` (no parallelization). To change this default and use a user-defined core number by default (say 6), set

In [ ]:
scqubits.settings.NUM_CPUS = 6

## Limiting BLAS threads per worker (`MULTIPROC_BLAS_THREADS`)


As noted at the top of this page, Numpy/Scipy internally multi-thread their linear algebra (through the BLAS backend), which competes with process-level parallelization: with `num_cpus` worker processes each spawning a full BLAS thread pool, the cores become oversubscribed and a sweep can run *slower* than with fewer threads per worker.

scqubits caps the per-worker BLAS threads automatically while the worker pool is created. The cap is controlled by `MULTIPROC_BLAS_THREADS`, which defaults to `"auto"`:

In [ ]:
# "auto" (default): cap each worker to max(1, cores // num_cpus) -- no oversubscription
# a positive int : a fixed per-worker cap, e.g. 1 for many small diagonalizations
# None           : opt out and leave the thread environment untouched
scqubits.settings.MULTIPROC_BLAS_THREADS = "auto"

With the default `"auto"`, each worker is capped to `max(1, cores // num_cpus)`, so the workers together use about one thread per core and never oversubscribe. A positive integer sets a fixed per-worker cap (e.g. `1` for many small diagonalizations), and `None` opts out, leaving the thread environment untouched. The cap affects only the worker pool; the parent process's environment and BLAS thread count are restored once the pool has been built, so serial work (`num_cpus = 1`) is never affected.

How the cap reaches the workers depends on the platform:

- **Spawn-based workers** (macOS and Windows) re-read the environment when they re-import Numpy/Scipy, so the cap applies directly.
- **Fork-based workers** (Linux) inherit the parent's already-initialized BLAS pool and ignore the environment variables. For these, scqubits uses [`threadpoolctl`](https://github.com/joblib/threadpoolctl) (a scqubits dependency) to reduce the parent's BLAS thread count for the duration of pool creation, so the forked workers inherit it.
- It has **no effect** when Numpy's BLAS exposes no thread control, as with Apple Accelerate on Apple Silicon. Note, however, that Scipy ships its own OpenBLAS there, so the cap still limits the threads used by Scipy's eigensolvers -- which is what most scqubits diagonalization relies on.

If the cap cannot take effect on your platform, scqubits emits a one-time warning; in that case you can fall back to exporting `OMP_NUM_THREADS`/`OPENBLAS_NUM_THREADS` (etc.) in the shell *before* importing Numpy.

## Worker-pool reuse


Within a single computation that issues several parallel `map` calls — for example a `ParameterSweep`, which sweeps each bare subsystem and then the dressed system — scqubits caches the worker pool in `scqubits.settings.POOL` and reuses it whenever the requested core count and backend match, instead of starting a fresh pool each time. The cached pool is shut down automatically at interpreter exit. This is transparent and requires no user action.


## Process start method (`fork` vs `spawn`)

How worker processes are created — the *start method* — is determined by your platform.
There is exactly one safe choice per platform, so scqubits selects it automatically; it is
not a user setting:

| platform | start method | why |
|---|---|---|
| Linux | `fork` | fast, and fork is safe |
| macOS | `spawn` | fork-after-threads is **unsafe** on macOS — Apple's Accelerate/GCD and the Objective-C runtime are not fork-safe, so forking a worker pool after the numerics have started threads can crash, deadlock, or hang. CPython itself defaults macOS to `spawn` since 3.8. This applies to **both Intel and Apple Silicon** Macs. |
| Windows | `spawn` | the only option |

The only consequence you need to be aware of is the `__main__` guard, below.

### The `__main__` guard (`spawn`/`forkserver` only)

With `spawn` (and `forkserver`), each worker process **re-imports your program's entry
module**. A **plain script** that triggers `num_cpus > 1` must therefore guard its entry
point, or the workers would re-run the script and Python raises a `RuntimeError`:

```python
import scqubits as scq

if __name__ == "__main__":
    sweep = scq.ParameterSweep(..., num_cpus=4)
```

**Jupyter/IPython need no guard.** scqubits emits a one-time warning the first time it
starts a `spawn` pool outside IPython, reminding you of this requirement.


### Cost

`spawn` workers re-import numpy/scipy/scqubits, so the **first** parallel sweep of a
session pays a one-time startup of roughly a second. Because the pool is cached and
reused (see *Worker-pool reuse* above), **every subsequent sweep is as fast as fork** — the
cost is paid once per session, not per sweep. For the heavy sweeps where `num_cpus > 1` is
worthwhile, this is negligible.

> **Note:** unlike fork children, `spawn` workers are not automatically reaped if the
> parent process is killed with `SIGKILL` mid-run, and may linger. A normal exit (or a
> `ParameterSweep.run()` completing) cleans them up.


## multiprocessing vs. pathos

scqubits supports parallelization through `multiprocessing` as well as `pathos`. The latter is the default option and is more robust thanks to the advanced pickling methods enabled through `dill`.

To switch from use of `pathos`/`dill` to `multiprocessing`, simply alter the following setting:

In [ ]:
scqubits.settings.MULTIPROC = 'multiprocessing'